In [1]:
import os
import sys

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [2]:
from hummingbot.strategy_v2.utils.distributions import Distributions
from controllers.directional_trading.pz_scalper import PZScalperControllerConfig
from core.backtesting.optimizer import BacktestingConfig, BaseStrategyConfigGenerator
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from decimal import Decimal


class PZMMConfigGenerator(BaseStrategyConfigGenerator):
    """
    Strategy configuration generator for PZ MM optimization.
    """
    async def generate_config(self, trial) -> BacktestingConfig:

        # Those doesn't matter, they are dyncamically calculated inside the controller anyway
        take_profit = 5 # trial.suggest_float("take_profit", 0.01, 0.03, step=0.01)
        stop_loss = 5  #trial.suggest_float("stop_loss", 0.01, 0.05, step=0.01)
        trailing_stop_activation_price = 1.0
        trailing_delta_ratio = 0.05 

        # Controller configuration
        connector_name = "binance_perpetual"
        trading_pair = "WLD-USDT"
        total_amount_quote = 1000
 

        trailing_stop_trailing_delta = trailing_stop_activation_price * trailing_delta_ratio
        time_limit = trial.suggest_int("time_limit", 300, 300 * 5, step=300)
        cooldown_time = 1 #trial.suggest_int("cooldown_time", 60, 60 * 5, step=60)
        
        
        hma_slow = trial.suggest_int("hma_slow", 20, 50, step = 5)
        hma_fast = trial.suggest_int("hma_fast", 10, 30, step = 5)
        natr_length = trial.suggest_int("natr_length", 7, 21, step = 2)
        interval = "5m"

        tp_natr_factor = trial.suggest_float("tp_natr_factor", 0.25, 3, step=0.25)
        sl_natr_factor = trial.suggest_float("sl_natr_factor", 0.5, 3, step=0.5)
        ts_activation_natr_factor = trial.suggest_float("ts_activation_natr_factor", 0.25, 1, step=0.25)
        ts_delta_natr_factor = trial.suggest_float("ts_delta_natr_factor", 0.25, 1, step=0.25)
        max_executors_per_side = 2


        # Creating the instance of the configuration and the controller
        config = PZScalperControllerConfig(
            connector_name=connector_name,
            trading_pair=trading_pair,
            interval=interval,
            take_profit=Decimal(take_profit),
            stop_loss=Decimal(stop_loss),
            trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
            total_amount_quote=Decimal(total_amount_quote),
            time_limit=time_limit,
            max_executors_per_side=max_executors_per_side,
            cooldown_time=cooldown_time,
            natr_length = natr_length,
            sl_natr_factor=sl_natr_factor,
            ts_activation_natr_factor = ts_activation_natr_factor,
            ts_delta_natr_factor = ts_delta_natr_factor,
            tp_natr_factor=tp_natr_factor,
            hma_fast=hma_fast,
            hma_slow=hma_slow,
        )

        # Return the configuration encapsulated in BacktestingConfig
        return BacktestingConfig(config=config, start=self.start, end=self.end)

In [3]:
from core.backtesting.optimizer import StrategyOptimizer
optimizer = StrategyOptimizer(root_path=root_path)

2025-04-08 17:17:09,989 - root - ERROR - Error writing configs: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'
Traceback (most recent call last):
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/client/config/config_helpers.py", line 876, in save_to_yml
    with open(yml_path, "w", encoding="utf-8") as outfile:
FileNotFoundError: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'


In [4]:
optimizer.launch_optuna_dashboard()

In [ ]:
import datetime

start_date = datetime.datetime(2025, 3, 20)
end_date = datetime.datetime(2025, 3, 30)
# start_date = datetime.datetime(2024, 8, 2)
# end_date = datetime.datetime(2024, 8, 3)
config_generator = PZMMConfigGenerator(start_date=start_date, end_date=end_date)

await optimizer.optimize(
    study_name="pz_scalper",
    config_generator=config_generator,
    n_trials=100,
)

[I 2025-04-08 17:17:10,765] A new study created in RDB with name: pz_scalper
2025-04-08 17:17:11,265 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x73f6da2a74f0>
2025-04-08 17:17:11,267 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x73f6f02b7280>, 82303.79591695)])']
connector: <aiohttp.connector.TCPConnector object at 0x73f6da2a7b80>
Listening on http://127.0.0.1:8080/
Hit Ctrl-C to quit.

Traceback (most recent call last):
  File "/home/pascal/anaconda3/envs/quants-lab/bin/optuna-dashboard", line 8, in <module>
    sys.exit(main())
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/optuna_dashboard/_cli.py", line 140, in main
    run_wsgiref(app, args.host, args.port, args.quiet)
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/optuna_dashboard/_cli.py", line 43, in run_wsgiref
    httpd = make_server(host, port, a